# DQMBot — Batch Image Query Driver

**Image layout expected:**
```
images/
    <subsystem>_<plotNumber>_<titleSlug>/
        <stem>[_grpN]_run<XXXXXX>.png
```

**Reference image layout:**
```
ref_images/
    <subsystem>/
        <subsystem>_<plotNumber>_<titleSlug>/
            <stem>[_grpN]_run<XXXXXX>.png
```

**Output layout (with run_id — preserves previous runs):**
```
results/
    <run_id>/
        <stem>/
            <stem>_<model>_run<XXXXXX>.txt
        summary_<run_id>.csv
```

**Output layout (no run_id — overwrites):**
```
results/
    <stem>/
        <stem>_<model>_run<XXXXXX>.txt
    summary.csv
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    ModelConfig, RunMetadata,
    list_models, build_messages, send_query, query,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
    find_reference_image, find_reference_images,
)
from rag_backends import (
    LocalRAG, YAMLContext, OWUIContext, NoContext,
    retrieve_and_inspect,
)

print('owui_client loaded OK')

In [ ]:
# ── Dependencies (run once, then restart kernel) ───────────────────────────────
# !pip install rank-bm25 langchain-huggingface sentence-transformers --quiet

In [ ]:
# ── Discover available models ──────────────────────────────────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

In [ ]:
# ── Plot preparation ──────────────────────────────────────────────────────────
RUN_ID      = 'localRAG'
OUTPUT_ROOT = Path('results')

IMAGE_ROOT = Path('images')   # must contain per-plot subdirs directly; nested roots not supported
# IMAGE_ROOT = Path('images/Ecal_03_Occupancy')
REF_DIR    = Path('ref_images')   # layout: ref_images/<subsystem>/<folder>/; None to disable

PLOT_FILTER = None                         # None — run all plots
# PLOT_FILTER = ['L1T_']                  # subsystem prefix
# PLOT_FILTER = ['L1T_01_', 'Ecal_00_']  # specific plots

_all_pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
pairs = [(p, img) for p, img in _all_pairs
         if not PLOT_FILTER or any(f in p for f in PLOT_FILTER)]
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── RAG / query config ────────────────────────────────────────────────────────

# Models: use a plain string, or ModelConfig to set Gemma 4 image token budget
# Valid image_token_budget values: 70, 140, 280, 560, 1120
MODELS = [
    # 'qwen2.5vl:latest',             #7b
    # 'qwen2.5vl:32b',                #32b
    # 'qwen3-vl:latest',              #8b
    # 'litellm-ow.qwen/qwen3.6',      #35b
    # 'gemma3:latest',                #4b
    'litellm-ow.google/gemma4-31b',                                          #31b, default budget
    # ModelConfig(name='litellm-ow.google/gemma4-31b', image_token_budget=1120),  #31b, high budget
]

# Pick one context backend:
CONTEXT = LocalRAG(csv_path=Path('document_chunks.csv'))   # BM25 + vector search over CSV
# CONTEXT = YAMLContext()                                   # direct lookup from plot_instructions/
# CONTEXT = OWUIContext(collection_ids=['<id>'])            # server-side retrieval via OWUI
# CONTEXT = NoContext()                                     # no retrieval

SYSTEM_PROMPT = (
 """\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad.

In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction
 - Decide if the plot is good or bad\
"""
)

PROMPT = (
    ''
)

DELAY = 1.5
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Optional: per-image prompt via run → event type mapping ──────────────────
# Uncomment and populate event_type_map to inject the event type into each
# per-image prompt (e.g. 'The plot is from a "collisions" event.').

# RUN_METADATA = None   # no per-run context

RUN_METADATA = RunMetadata(event_type_map={
    398185: 'collisions',
    398186: 'cosmics',
    398187: 'circulating',
    398188: 'collisions',
    398189: 'collisions',
    398191: 'collisions',
    398194: 'cosmics',
    398199: 'cosmics',
})

In [ ]:
# ── Debug: inspect what RAG retrieves for the query ──────────────────────────
# Only applies when CONTEXT is a LocalRAG instance.
debug_query = "ECal TP ET-weighted Occupancy"
hits = retrieve_and_inspect(debug_query, csv_path=CONTEXT.csv_path, top_k=5, method="hybrid")

print(f'Query: "{debug_query}"\n')
for h in hits:
    print(f"  #{h['rank']}  score={h['score']:.4f}  doc={h['document_id']}  "
          f"chunk={h['chunk_index']}  len={h['text_length']}")
    print(f"       source: {h['source']}")
    print(f"       preview: {h['text_preview'][:120]}...")
    print()

In [ ]:
# ── Sanity check: show what will be processed and where it will land ──────────
plot_names = sorted(set(p for p, _ in pairs))

print(f'Plots found  : {len(plot_names)}')
for pn in plot_names:
    imgs = [img for p, img in pairs if p == pn]
    print(f'  {pn}/  ({len(imgs)} images)')
    for img in imgs:
        refs = find_reference_images(img, REF_DIR) if REF_DIR and REF_DIR.exists() else []
        if refs:
            for r in refs:
                print(f'    {img.name}  ← {r.name}')
        else:
            print(f'    {img.name}  ← (no reference)')

if REF_DIR and REF_DIR.exists():
    covered = sum(1 for _, img in pairs if find_reference_images(img, REF_DIR))
    print(f'\nReference dir: {REF_DIR}/  ({covered}/{len(pairs)} images have references)')
else:
    print(f'\nReference dir: {REF_DIR} — not found, reference images will be skipped')

print(f'\nModels       : {len(MODELS)}')
for m in MODELS:
    print(f'  {m}')

print(f'\nRun ID       : {RUN_ID or "(none — overwrite mode)"}')
print(f'Total queries: {len(pairs) * len(MODELS)}')

print('\nExample output paths:')
for model in MODELS:
    plot_name, img = pairs[0]
    d = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
    f = resolve_output_file(d, img, model)
    print(f'  {f}')

In [ ]:
# ── Smoke test: one image, first model ───────────────────────────────────────
if pairs:
    plot_name, img = pairs[0]
    refs = find_reference_images(img, REF_DIR) if REF_DIR and REF_DIR.exists() else []

    _m = MODELS[2]
    smoke_model = _m if isinstance(_m, ModelConfig) else ModelConfig(name=_m)

    spec = build_messages(
        PROMPT,
        system=SYSTEM_PROMPT,
        reference_images=refs,
        image_path=img,
        context=CONTEXT,
    )

    print(f"Plot            : {plot_name}")
    print(f"Image           : {img.name}")
    print(f"References      : {[r.name for r in spec['ref_list']]}")
    print(f"RAG backend     : {type(spec['context']).__name__}")
    print(f"RAG text ({len(spec['rag_text'])} chars):")
    print(spec['rag_text'][:600] or '  (none)')
    print()

    test = send_query(spec, model=smoke_model)
    print(f"Model      : {test['model']}")
    print(f"Load       : {test['load_latency_s']}s")
    print(f"Generation : {test['generation_latency_s']}s")
    print(f"Total      : {test['latency_s']}s")
    print(f"Error      : {test['error']}")
    print()
    print(test['response'])

### Run on all images

In [ ]:
# ── Full batch ────────────────────────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

results = batch_query_images(
    PROMPT,
    image_root=IMAGE_ROOT,
    models=MODELS,
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
    system=SYSTEM_PROMPT,
    ref_dir=REF_DIR if REF_DIR and REF_DIR.exists() else None,
    context=CONTEXT,
    pairs=pairs,
    delay=DELAY,
    verbose=True,
    run_metadata=RUN_METADATA,
)

print(f'\nDone. {len(results)} queries completed.')

In [ ]:
# ── Retry errors ──────────────────────────────────────────────────────────────
failed = [
    (r['model'], r['image'])
    for r in results if r['error'] is not None
]

if not failed:
    print('No errors in results — nothing to retry.')
else:
    print(f'Retrying {len(failed)} failed quer{"y" if len(failed)==1 else "ies"}...')
    retry_results = []

    for i, (model_name, image_str) in enumerate(failed):
        image_path = Path(image_str)
        plot_name  = image_path.parent.name
        print(f'  [{i+1}/{len(failed)}] model={model_name}  image={image_path.name} ...', end=' ', flush=True)

        result = query(
            PROMPT,
            model=ModelConfig(name=model_name),
            system=SYSTEM_PROMPT,
            image_path=image_path,
            context=CONTEXT,
        )
        result['plot_name'] = plot_name
        retry_results.append(result)

        out_dir  = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = resolve_output_file(out_dir, image_path, model_name)
        with open(out_file, 'w') as f:
            f.write(f"Model:      {result['model']}\n")
            f.write(f"Plot:       {plot_name}\n")
            f.write(f"Image:      {result['image']}\n")
            refs_str = ', '.join(Path(r).name for r in result['reference_images'])
            f.write(f"References: {refs_str or '(none)'}\n")
            f.write(f"Run ID:     {RUN_ID or '(overwrite)'}\n")
            f.write(f"Load latency:       {result['load_latency_s']}s\n")
            f.write(f"Generation latency: {result['generation_latency_s']}s\n")
            f.write(f"Total latency:      {result['latency_s']}s\n")
            f.write(f"Prompt:     {result['prompt']}\n")
            f.write('-' * 60 + '\n')
            if result['error']:
                f.write(f"ERROR: {result['error']}\n")
            else:
                f.write(result['response'] + '\n')

        status = 'ERROR' if result['error'] else f"{result['latency_s']}s → {out_file}"
        print(status)
        time.sleep(DELAY)

    retry_index = {(r['model'], r['image']): r for r in retry_results}
    results = [
        retry_index.get((r['model'], r['image']), r)
        for r in results
    ]

    still_failing = sum(1 for r in retry_results if r['error'])
    print(f'\nDone. {len(retry_results) - still_failing}/{len(retry_results)} recovered.')
    if still_failing:
        print('Still failing:')
        for r in retry_results:
            if r['error']:
                print(f"  {r['model']}  {Path(r['image']).name}  → {r['error']}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import re as _re

df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model'])[['load_latency_s', 'generation_latency_s', 'latency_s']]
      .mean()
      .round(2)
      .rename(columns={'load_latency_s': 'load_s', 'generation_latency_s': 'gen_s', 'latency_s': 'total_s'})
)

In [ ]:
# ── Save CSV next to the run's output folder ──────────────────────────────────
if RUN_ID:
    csv_path = OUTPUT_ROOT / RUN_ID / f'summary_{RUN_ID}.csv'
else:
    csv_path = OUTPUT_ROOT / 'summary.csv'

df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Show result tree
print()
root = OUTPUT_ROOT / RUN_ID if RUN_ID else OUTPUT_ROOT
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

In [ ]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = plot_names[0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model_used']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    print(row['error'] and f"ERROR: {row['error']}" or row['response'])
    print()